## 在 TinyStories 上预训练 GPT-2，然后做关键部件的消融

上半节手写了 GPT-2。这一节让它真正学东西：

1. **数据**：TinyStories 语料 + 我们自己训的 4096 词表
2. **训练**：一个完整的预训练循环，混合精度、warmup、梯度裁剪
3. **消融**：把 GPT-2 的部件一个个拆掉，看每个部件在防止什么样的失败
4. **长训 + 生成**：训一个能写故事的模型，比较不同的采样方法



In [1]:
!pip -q install tokenizers huggingface_hub

import math, time
from dataclasses import dataclass, replace
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

device = "cuda" if torch.cuda.is_available() else "cpu"
print(torch.__version__, device, torch.cuda.get_device_name(0) if device == "cuda" else "")

HF_REPO = "你的用户名/tinystories-4096"     # 数据和 checkpoint 放在这个 HuggingFace dataset 仓库
VOCAB = 4096
EOT = "<|endoftext|>"

2.11.0+cpu cpu 


---
## 1. 准备 TinyStories 语料和 4096 词表

这份 notebook 给想要在真实语料上多步训练（包括训练Tokenizer）的你。lab1里已经说了，我们写的12层模型和GPT-2不完全一样，但是仍然能进行正常的梯度下降记住一些随机序列。这份代码产出三个文件，你可以用他们进行预训练：

- `tokenizer.json`：在 TinyStories 上训练的 byte-level BPE，词表 4096
- `train.bin`：训练集，每篇故事后面接一个 `<|endoftext|>`，全部首尾相接拼成一条长序列，用 `uint16` 存，每个 token 2 字节。
- `val.bin`：验证集，格式同上

byte-level BPE和 GPT-2 的 tokenizer 是同一种算法：

1. 先把文本拆成字节，256 种字节就是初始词表，任何文本都能表示，不会有未知字符
2. 统计相邻两个 token 一起出现的次数，把最常见的一对合并成一个新 token
3. 重复，直到词表涨到 4096

GPT-2 在网页文本上合并到 50257；TinyStories 用词简单，4096 已经能把绝大多数常用词合并成单个 token。

In [ ]:
N_TRAIN, N_VAL, N_TOK_TRAIN = 200_000, 5_000, 50_000      # 训练集 / 验证集 / 训练词表用的故事数

def read_stories(path, n):
    stories, buf = [], ""
    with open(path, encoding="utf-8") as f:
        while len(stories) < n:
            chunk = f.read(1 << 20)                       # 每次读 1MB，读够就停，不用把 2GB 全读进来
            if not chunk:
                break
            parts = (buf + chunk).split(EOT)
            buf = parts.pop()                             # 最后一段可能没读完，留到下一轮
            stories += [s.strip() for s in parts if s.strip()]
    if len(stories) < n and buf.strip():
        stories.append(buf.strip())
    return stories[:n]

def train_tokenizer(stories):
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=VOCAB, special_tokens=[EOT], min_frequency=2,
                                  initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
    tok.train_from_iterator(stories, trainer)
    return tok

def encode_all(tok, stories, path, bs=10_000):
    eot_id, chunks, lens = tok.token_to_id(EOT), [], []
    for i in range(0, len(stories), bs):
        for enc in tok.encode_batch(stories[i:i + bs]):          # Rust 里多线程并行编码
            lens.append(len(enc.ids))
            chunks.append(np.array(enc.ids + [eot_id], dtype=np.uint16))
    arr = np.concatenate(chunks)
    assert arr.max() < VOCAB
    arr.tofile(path)
    lens = np.array(lens)
    print(f"{path}: {len(arr):,} token，{arr.nbytes / 1e6:.0f} MB；每篇平均 {lens.mean():.0f} token，"
          f"中位数 {np.median(lens):.0f}，超过 256 的占 {(lens > 256).mean():.1%}")

t = time.time()
train_txt = hf_hub_download("roneneldan/TinyStories", "TinyStoriesV2-GPT4-train.txt", repo_type="dataset")
val_txt = hf_hub_download("roneneldan/TinyStories", "TinyStoriesV2-GPT4-valid.txt", repo_type="dataset")
train_stories, val_stories = read_stories(train_txt, N_TRAIN), read_stories(val_txt, N_VAL)
print(f"读入 train {len(train_stories):,} 篇，val {len(val_stories):,} 篇，用时 {time.time() - t:.0f} s")

tok = train_tokenizer(train_stories[:N_TOK_TRAIN])
tok.save("tokenizer.json")
for s in [train_stories[1][:200], "Lily and Tom went to the park. They saw a big, red ball!"]:
    assert tok.decode(tok.encode(s).ids) == s                # 编码再解码能原样还原
print("词表", tok.get_vocab_size(), "最后合并的 token：",
        [t for t, _ in sorted(tok.get_vocab().items(), key=lambda kv: kv[1])[-15:]])

encode_all(tok, train_stories, "train.bin")
encode_all(tok, val_stories, "val.bin")


"超过 256 的占多少"决定上下文 256 够不够用：超过的故事训练时会被截成几段，模型看不到完整的故事。

---
## 2. 加载数据

`np.memmap` 不把文件读进内存，用到哪段读哪段。每次随机截 257 个 token：前 256 个是输入，往后错一位的 256 个是目标，每个位置都在预测下一个 token。

In [ ]:
tok = Tokenizer.from_file("tokenizer.json")
EOT_ID = tok.token_to_id(EOT)
data = {split: np.memmap(f"{split}.bin", dtype=np.uint16, mode="r") for split in ["train", "val"]}
print({s: f"{len(d):,} token" for s, d in data.items()})

def get_batch(split, B=32, T=256):
    d = data[split]
    ix = np.random.randint(0, len(d) - T - 1, B)
    x = torch.from_numpy(np.stack([d[i:i + T] for i in ix]).astype(np.int64))
    y = torch.from_numpy(np.stack([d[i + 1:i + 1 + T] for i in ix]).astype(np.int64))
    if device == "cuda":
        return x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    return x, y

x, y = get_batch("train")
assert torch.equal(x[:, 1:], y[:, :-1])
print(x.shape, "\n---- 随机一段 ----\n" + tok.decode(x[0].tolist(), skip_special_tokens=False)[:500])


## 3. 模型

和上半节手写的 GPT-2 完全一样，只多了几个开关，后面消融用：

| 开关 | 默认（GPT-2） | 关掉之后 |
|---|---|---|
| `causal` | 只能看前面 | 每个位置能看到整条序列，包括未来 |
| `pos_emb` | 加位置 embedding | 不加 |
| `scale` | 分数除以 $\sqrt d$ | 不除 |
| `residual` | $x + f(x)$ | 只有 $f(x)$ |
| `ln` | `"pre"`：$x + f(\text{LN}(x))$ | `"post"`：$\text{LN}(x + f(x))$，原版 Transformer 的写法 |

attention 这里换成了 PyTorch 自带的 `F.scaled_dot_product_attention`，算的东西和上半节手写的一样（上半节验证过），但快很多、省显存：它不会把 $(B, H, T, T)$ 的 `att` 整个存下来。

In [ ]:
@dataclass
class GPTConfig:
    block_size: int = 256
    vocab_size: int = VOCAB
    n_layer: int = 12
    n_head: int = 8
    n_embd: int = 256
    causal: bool = True
    pos_emb: bool = True
    scale: bool = True
    residual: bool = True
    ln: str = "pre"            # "pre" 或 "post"

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.c_attn = nn.Linear(cfg.n_embd, 3 * cfg.n_embd)
        self.c_proj = nn.Linear(cfg.n_embd, cfg.n_embd)

    def forward(self, x):
        B, T, C = x.shape
        H = self.cfg.n_head
        q, k, v = (t.view(B, T, H, C // H).transpose(1, 2) for t in self.c_attn(x).split(C, dim=2))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=self.cfg.causal,
                                           scale=None if self.cfg.scale else 1.0)   # None 表示除以 √d
        return self.c_proj(y.transpose(1, 2).contiguous().view(B, T, C))

class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.c_fc = nn.Linear(cfg.n_embd, 4 * cfg.n_embd)
        self.gelu = nn.GELU(approximate="tanh")
        self.c_proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd)
    def forward(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.ln_1, self.attn = nn.LayerNorm(cfg.n_embd), CausalSelfAttention(cfg)
        self.ln_2, self.mlp = nn.LayerNorm(cfg.n_embd), MLP(cfg)

    def forward(self, x):
        if self.cfg.ln == "post":                           # 原版 Transformer：先加残差，再 LN
            x = self.ln_1(x + self.attn(x))
            return self.ln_2(x + self.mlp(x))
        a = self.attn(self.ln_1(x))                         # GPT-2：pre-LN
        x = x + a if self.cfg.residual else a
        m = self.mlp(self.ln_2(x))
        return x + m if self.cfg.residual else m

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(cfg.vocab_size, cfg.n_embd),
            wpe=nn.Embedding(cfg.block_size, cfg.n_embd),
            h=nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)]),
            ln_f=nn.LayerNorm(cfg.n_embd),
        ))
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.transformer.wte.weight     # 权重共享
        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):                # 写回残差流的投影，std 再除以 √(2L)
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.transformer.wte(idx)
        if self.cfg.pos_emb:
            x = x + self.transformer.wpe(torch.arange(T, device=idx.device))
        for block in self.transformer.h:
            x = block(x)
        logits = self.lm_head(self.transformer.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, greedy=False):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.cfg.block_size:])
            logits = logits[:, -1].float()
            if greedy:
                nxt = logits.argmax(-1, keepdim=True)
            else:
                logits = logits / temperature
                if top_k is not None:
                    kth = torch.topk(logits, top_k).values[:, -1:]
                    logits = logits.masked_fill(logits < kth, float("-inf"))
                nxt = torch.multinomial(logits.softmax(-1), 1)
            idx = torch.cat([idx, nxt], dim=1)
            if idx.size(0) == 1 and nxt.item() == EOT_ID:           # 写完一篇故事就停
                break
        return idx

print(f"参数量 {sum(p.numel() for p in GPT(GPTConfig()).parameters()):,}")

注意 loss 那一行的 `logits.float()`：混合精度下 logits 是 fp16，只有 3 位左右有效数字，最大只能表示 65504。交叉熵要对 4096 个 $e^{z}$ 求和再取 log，放在 fp32 里算更稳。

---
## 4. 训练循环

和 lab0 的 `fit` 是同一个循环，多了四样大模型训练的标配：

- **混合精度（fp16）**：T4 是 Turing 架构，不支持 bf16，只能用 fp16。矩阵乘法用 fp16 算，快一倍多；参数和优化器状态仍是 fp32
- **GradScaler**：fp16 能表示的最小正数约 $6 \times 10^{-8}$，很多梯度比这还小，会直接变成 0（下溢）。做法是先把 loss 乘一个大数（比如 $2^{16}$）再反向传播，梯度跟着放大，更新参数前再除回去。出现 inf 就跳过这一步、把倍数调小
- **warmup + cosine**：学习率前 `warmup` 步从 0 线性升上去，之后按余弦降到 1/10。刚初始化时梯度方向很乱，Adam 对二阶矩的估计也还不准，一上来就用大学习率容易冲飞
- **梯度裁剪**：梯度总范数超过 1 就等比例缩小，防止偶尔一个坏 batch 把参数推得太远

weight decay （一种收缩权重的正则化手段）只加在矩阵上，不加在 bias 和 LayerNorm 上（它们是一维的）。

训练时每 `log_every` 步打印四个数，出问题时先看它们：

- **grad**：裁剪前的梯度范数。突然变大说明要发散了；长期远大于 1 说明裁剪一直在起作用，学习率可能太大
- **param**：参数范数。一直涨说明权重在变大，weight decay 就是压它的
- **out ent**：模型输出的下一个词分布的熵。刚开始约等于 $\ln V = 8.3$（瞎猜），之后越来越确定
- **attn ent**：attention 权重的熵（各层平均）。太低说明 attention 塌到了一两个位置上；6.3 节专门看它

In [ ]:
@torch.no_grad()
def estimate_loss(model, split, iters=20, B=32):
    model.eval()
    losses = []
    for _ in range(iters):
        x, y = get_batch(split, B, model.cfg.block_size)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):
            losses.append(model(x, y)[1].item())
    model.train()
    return float(np.mean(losses))

@torch.no_grad()
def attn_entropy(model, B=8):
    """各层 attention 权重的平均熵。只看后半段的位置：前面的位置能看的 token 太少，熵天然就低"""
    cfg = model.cfg
    x, _ = get_batch("val", B, cfg.block_size)
    T = x.size(1)
    h = model.transformer.wte(x)
    if cfg.pos_emb:
        h = h + model.transformer.wpe(torch.arange(T, device=x.device))
    future = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), 1)
    ents = []
    for blk in model.transformer.h:
        a = blk.ln_1(h) if cfg.ln == "pre" else h               # 送进 attention 的就是这个
        q, k, _ = blk.attn.c_attn(a).split(cfg.n_embd, dim=2)
        q, k = (t.view(B, T, cfg.n_head, -1).transpose(1, 2) for t in (q, k))
        s = q @ k.transpose(-2, -1)
        if cfg.scale:
            s = s / math.sqrt(q.size(-1))
        if cfg.causal:
            s = s.masked_fill(future, float("-inf"))
        p = s.float().softmax(-1)[:, :, T // 2:]
        ents.append(-(p * p.clamp_min(1e-12).log()).sum(-1).mean().item())
        h = blk(h)
    return float(np.mean(ents))

def uniform_attn_entropy(T=256):                                # 对照：每个位置在能看的范围内均匀分配
    return float(np.mean(np.log(np.arange(T // 2, T) + 1)))

def train(cfg, steps, lr=1e-3, warmup=100, B=32, eval_every=None, seed=0, label="", qk_std=None, log_every=100):
    print(f"训练 {label}，{steps} 步")
    torch.manual_seed(seed); np.random.seed(seed)
    model = GPT(cfg).to(device)
    if qk_std is not None:                                     # 只重设 q、k 的投影（c_attn 的前 2C 行），v 不动
        with torch.no_grad():
            for b in model.transformer.h:
                nn.init.normal_(b.attn.c_attn.weight[:2 * cfg.n_embd], 0.0, qk_std)
    decay = [p for p in model.parameters() if p.dim() >= 2]
    no_decay = [p for p in model.parameters() if p.dim() < 2]
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": 0.1}, {"params": no_decay, "weight_decay": 0.0}],
                            lr=lr, betas=(0.9, 0.95))
    use_amp = device == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    def lr_at(s):
        if s < warmup:
            return lr * (s + 1) / warmup
        p = (s - warmup) / max(1, steps - warmup)
        return lr * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * p)))

    hist = {"label": label, "step": [], "train": [], "val_step": [], "val": [], "attn_step": [], "attn_ent": []}
    model.train()
    t0 = time.time()
    for s in range(steps):
        for g in opt.param_groups:
            g["lr"] = lr_at(s)
        x, y = get_batch("train", B, cfg.block_size)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits, loss = model(x, y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()           # 先把 loss 乘以一个大的系数（比如 65536），再 backward
        scaler.unscale_(opt)                    # 先除回去，再裁剪
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()

        if s % 10 == 0 or s == steps - 1:      # 画图用：记得密一点
            hist["step"].append(s); hist["train"].append(loss.item())
        if s % log_every == 0 or s == steps - 1:   # 打印用：这些指标要额外算，只在打印时算
            with torch.no_grad():
                lp = F.log_softmax(logits.float(), dim=-1)
                out_ent = -(lp.exp() * lp).sum(-1).mean().item()
                param_norm = torch.linalg.vector_norm(torch.stack([p.float().norm() for p in model.parameters()])).item()
            ae = attn_entropy(model)
            hist["attn_step"].append(s); hist["attn_ent"].append(ae)
            print(f"  step {s:6d}  loss {loss.item():.3f}  grad {grad_norm.item():7.3f}  param {param_norm:7.2f}"
                  f"  out ent {out_ent:.2f}  attn ent {ae:.2f}")
        if eval_every and (s % eval_every == 0 or s == steps - 1):
            hist["val_step"].append(s); hist["val"].append(estimate_loss(model, "val"))
            print(f"  step {s:6d}  train {loss.item():.3f}  val {hist['val'][-1]:.3f}  {time.time() - t0:6.0f} s")
    sec = time.time() - t0
    hist["val_final"] = estimate_loss(model, "val")
    hist["sec"] = sec
    print(f"[{label}] {steps} 步 {sec:.0f} s（{sec / steps * 1e3:.0f} ms/step，{steps * B * cfg.block_size / sec:,.0f} token/s）"
          f"  最后 train {np.mean(hist['train'][-5:]):.3f}  val {hist['val_final']:.3f}\n")
    return model, hist

def smooth(v, a=0.9):
    out, m = [], None
    for x in v:
        m = x if m is None or not math.isfinite(m) else a * m + (1 - a) * x
        out.append(m)
    return out

def plot_runs(hists, title="", ylim=(1, 9), attn=False):
    fig, axes = plt.subplots(1, 2 if attn else 1, figsize=(14 if attn else 8, 4.5), squeeze=False)
    ax = axes[0]
    for h in hists:
        ax[0].plot(h["step"], smooth(h["train"], 0.7), lw=2, label=f"{h['label']}  (val {h['val_final']:.2f})")
    ax[0].axhline(math.log(VOCAB), ls=":", c="gray", lw=1, label="ln V (random guess)")
    ax[0].set_xlabel("step"); ax[0].set_ylabel("train loss"); ax[0].set_ylim(*ylim); ax[0].set_title(title); ax[0].legend()
    if attn:
        for h in hists:
            ax[1].plot(h["attn_step"], h["attn_ent"], "o-", lw=2, label=h["label"])
        ax[1].axhline(uniform_attn_entropy(), ls=":", c="gray", lw=1, label="uniform attention")
        ax[1].set_xlabel("step"); ax[1].set_ylabel("attention entropy"); ax[1].set_ylim(bottom=0); ax[1].legend()
    plt.show()

---
## 5. 基线

先用完整的 GPT-2 训 `ABL_STEPS` 步，之后每个消融都用**同样的步数、同样的随机种子、同样的数据顺序**，只改一个开关。

这一格也用来测速：看打印出来的 ms/step，算一下后面 6 个消融一共要多久。时间不够就把 `ABL_STEPS` 调小。

In [ ]:
ABL_STEPS = 600
runs, models = {}, {}

models["baseline"], runs["baseline"] = train(GPTConfig(), ABL_STEPS, label="baseline (GPT-2)")
plot_runs([runs["baseline"]], "baseline")

---
## 6. 消融

### 6.1 去掉 causal mask

PPT 里那张图上写的是 **Masked** multi-headed self-attention。拿掉 mask 会怎样？

In [ ]:
models["no_mask"], runs["no_mask"] = train(GPTConfig(causal=False), ABL_STEPS, label="no causal mask")
plot_runs([runs["baseline"], runs["no_mask"]], "no causal mask", ylim=(0, 9))

loss 降得比基线快得多，几乎到 0。验证集上也一样低。看起来是个更好的模型？让它写一段：

In [ ]:
prompt = torch.tensor([tok.encode("Once upon a time").ids], device=device)
for name in ["baseline", "no_mask"]:
    torch.manual_seed(0)
    out = models[name].eval().generate(prompt, 60, temperature=0.8, top_k=50)
    models[name].train()
    print(f"---- {runs[name]['label']} ----\n{tok.decode(out[0].tolist())}\n")


没有 mask，第 $t$ 个位置训练时能直接看到第 $t+1$ 个 token，也就是答案本身。它学会的是"把下一个位置的输入抄过来"，所以 loss 接近 0。

生成的时候没有下一个位置可抄：最后一个位置后面什么都没有，它从没学过这种情况该怎么办，于是只能乱写。

**训练时看到的信息必须和使用时一样多。** loss 低不代表模型好（可能存在偷看后面），评测方式和实际用法不一致，loss 再低也没有意义。

### 6.2 去掉位置 embedding

PPT 第 2 页的问题：第 1 个位置和第 7 个位置对第 8 个位置来说是一样的吗？

In [ ]:
models["no_pos"], runs["no_pos"] = train(GPTConfig(pos_emb=False), ABL_STEPS, label="no position embedding")
plot_runs([runs["baseline"], runs["no_pos"]], "no position embedding", ylim=(1, 9))

差距可能比你想的小。原因在 causal mask 上：第 $t$ 个位置能看到 $t$ 个 token，第 1 个位置只能看到自己。注意力在前面所有位置上平均，平均的范围本身就随位置变化，模型可以从中推断出"我大概在第几个"。有论文专门研究过这个现象：只用 causal mask、完全不加位置编码的语言模型，依然能学到位置信息（Haviv et al., 2022）。

但这种"推断出来的位置"很粗糙。回到 6.1 去掉了 mask 的模型：如果 mask 和位置 embedding 都没有，attention 对顺序就完全不敏感了，把句子打乱，每个位置的输出不变，说明模型就是靠泄露的下一个位置预测的。

同时设 `causal=False, pos_emb=False`，然后把输入序列打乱，检查输出是否只是跟着打乱。


In [ ]:
models["no_pos_no_mask"], runs["no_pos_no_mask"] = train(GPTConfig(pos_emb=False, causal=False), ABL_STEPS, label="no pos emb, no mask")
plot_runs([runs["baseline"], runs["no_mask"], runs["no_pos_no_mask"]], "no position embedding, no mask", ylim=(0, 9))

# 打乱输入顺序：输出是不是只是跟着打乱？
x, _ = get_batch("val", 4)
perm = torch.randperm(x.size(1), device=device)
for name in ["baseline", "no_pos_no_mask"]:
    m = models[name].eval()
    with torch.no_grad():
        a, b = m(x)[0], m(x[:, perm])[0]
    m.train()
    print(f"{runs[name]['label']:22s} 打乱输入后，输出与「原输出跟着打乱」的最大差 {(a[:, perm] - b).abs().max().item():.2e}")

没有位置 embedding 又没有 mask，模型对顺序完全不敏感：输入打乱，输出只是跟着打乱，差别只是浮点误差。

注意它的 loss 也降不到 0。它同样看得到下一个 token，但分不清哪个位置是"下一个"，所以抄不了答案，只能根据整段里有哪些词来猜。


### 6.3 去掉 $\sqrt d$

上节课推过：如果 $q, k$ 每一维方差为 1，$q \cdot k$ 的方差是 $d$，不除以 $\sqrt d$，softmax 会越来越接近 one-hot，梯度消失, 熵减小。先直接验证这件事：

In [ ]:
torch.manual_seed(0)
print(f"{'d':>6s} | {'除以√d':^20s} | {'不除':^20s}")
print(f"{'':>6s} | {'最大权重':>9s} {'熵':>9s} | {'最大权重':>9s} {'熵':>9s}")
for d in [16, 64, 256, 1024]:
    q, k = torch.randn(1000, d), torch.randn(64, d)            # 1000 个 query，64 个 key，每维方差 1
    row = []
    for s in [math.sqrt(d), 1.0]:
        p = (q @ k.T / s).softmax(-1)
        row += [p.max(-1).values.mean().item(), -(p * p.clamp_min(1e-12).log()).sum(-1).mean().item()]
    print(f"{d:6d} | {row[0]:9.3f} {row[1]:9.3f} | {row[2]:9.3f} {row[3]:9.3f}")
print(f"\n均匀分布的熵 ln 64 = {math.log(64):.3f}")

不除的话，$d=1024$ 时几乎每个 query 都把全部注意力压在一个 key 上。softmax 在这种状态下的梯度接近 0，模型改不动了。

再在真实模型上训一次：

In [ ]:
QK_STD = 1 / math.sqrt(256)       # LN 输出每一维方差为 1，q 的每一维方差 = C·std² = 1，正好满足上面推导的前提

# GPT-2 默认初始化（std 0.02）
models["h1"], runs["h1"] = train(GPTConfig(n_head=1), ABL_STEPS, label="sqrt(d), GPT-2 init")
models["h1_noscale"], runs["h1_noscale"] = train(GPTConfig(n_head=1, scale=False), ABL_STEPS, label="no sqrt(d), GPT-2 init")
# q、k 每一维方差为 1
models["h1_qk1"], runs["h1_qk1"] = train(GPTConfig(n_head=1), ABL_STEPS, qk_std=QK_STD, label="sqrt(d), q/k var 1")
models["h1_qk1_noscale"], runs["h1_qk1_noscale"] = train(GPTConfig(n_head=1, scale=False), ABL_STEPS, qk_std=QK_STD, label="no sqrt(d), q/k var 1")

plot_runs([runs["h1"], runs["h1_noscale"]], "GPT-2 init (std 0.02)", attn=True)
plot_runs([runs["h1_qk1"], runs["h1_qk1_noscale"]], "q/k variance 1", attn=True)

单头 256 维，用 GPT-2 默认初始化时，**loss 几乎看不出差别**，原因有两个：

1. **推导的前提不成立**。推导假设 $q, k$ 每一维方差为 1。但 GPT-2 的初始化 std 只有 0.02，$q$ 每一维的方差是 $C \cdot 0.02^2 = 256 \times 0.0004 \approx 0.1$，$q \cdot k$ 的标准差只有 $\sqrt{256} \times 0.1 \approx 1.6$，softmax 离饱和还远
2. **Adam 把变小的梯度放大回来了**。Adam 用每个参数自己的历史梯度幅度去除更新量，softmax 饱和让梯度变小，Adam 会自动补偿

但 **attention 熵**（右图）不一样：不除 $\sqrt d$ 的模型，attention 在训练中塌得更快、更低。loss 看不出来的东西，换个指标就看出来了。

把 $q, k$ 的初始化调到每一维方差为 1，推导的前提成立了：不除 $\sqrt d$ 的模型一开始 attention 就几乎全压在一个位置上，loss 也被拖累。

GPT-2 用小初始化，一开始碰不到这个问题。但训练中 $q, k$ 会慢慢变大，模型越宽、头的维度越大，问题越早出现。$\sqrt d$ 是一份不管尺度怎么变都成立的保险。

### 6.4 层数多了：残差和 LayerNorm 的位置

"一层不够，多来几层"。我们的模型已经是 12 层。先去掉残差：

In [ ]:
models["no_res"], runs["no_res"] = train(GPTConfig(residual=False), ABL_STEPS, label="no residual")
plot_runs([runs["baseline"], runs["no_res"]], "no residual (12 layers)", ylim=(1, 9))

没有残差时，第 1 层的梯度要穿过上面 11 层的 attention 和 MLP 才能回来，每穿过一层乘一次雅可比矩阵，乘 11 次之后要么很小要么很大。有了残差，$x + f(x)$ 对 $x$ 的导数是 $I + f'(x)$，里面那个 $I$ 让梯度有一条直通的路，层数再多也传得回来。

**可以试试**：`n_layer=2` 时再去掉残差，差距还大吗？

下面看 LayerNorm 放在哪里。原版 Transformer 是 post-LN，GPT-2 改成了 pre-LN。区别在残差那条直通的路上有没有 LN：

- post-LN：$x \leftarrow \text{LN}(x + f(x))$，梯度每过一层都要穿过一次 LN
- pre-LN：$x \leftarrow x + f(\text{LN}(x))$，直通的路上什么都没有

post-LN 的模型对学习率很敏感，一般靠 warmup 才能训起来。这里两种都**不加 warmup**，直接用 1e-3 起步：

In [ ]:
models["pre_nowarm"], runs["pre_nowarm"] = train(GPTConfig(), ABL_STEPS, warmup=0, label="pre-LN, no warmup")
models["post_nowarm"], runs["post_nowarm"] = train(GPTConfig(ln="post"), ABL_STEPS, warmup=0, label="post-LN, no warmup")
plot_runs([runs["baseline"], runs["pre_nowarm"], runs["post_nowarm"]], "LayerNorm position x warmup", ylim=(1, 9))

如果 post-LN 也正常训起来了，把两次的 `lr` 都改成 `3e-3` 再跑一次：模型越小越不容易出问题，要把学习率推高才能看出区别。

### 6.5 汇总

In [ ]:
print(f"{'':30s} {'val loss':>9s} {'用时':>7s}")
for name, h in runs.items():
    print(f"{h['label']:30s} {h['val_final']:9.3f} {h['sec']:6.0f}s")
plot_runs(list(runs.values()), "all ablations", ylim=(0, 9))

---
## 7. 长训，然后写故事

消融只训了几百步，模型还写不出像样的故事。这里可以选择训一个完整的版本。Colab 免费版会话可能中途断开，长训时注意保存。

In [ ]:
TRAIN_LONG = True      
UPLOAD = False        # 上传时改成 True（需要 HF write token）,不上传则为False.
LONG_STEPS = 10_000

if TRAIN_LONG:
    long_model, long_hist = train(GPTConfig(), LONG_STEPS, lr=1e-3, warmup=200, eval_every=1000, log_every=1000, label="long run")
    torch.save({"cfg": long_model.cfg.__dict__, "model": long_model.state_dict()}, "ckpt.pt")
    plot_runs([long_hist], "long run", ylim=(1, 9))
    if UPLOAD:
        from huggingface_hub import HfApi, login
        login()
        HfApi().create_repo(HF_REPO, repo_type="dataset", exist_ok=True)
        HfApi().upload_file(path_or_fileobj="ckpt.pt", path_in_repo="ckpt.pt", repo_id=HF_REPO, repo_type="dataset")
else:
    ck = torch.load(hf_hub_download(HF_REPO, "ckpt.pt", repo_type="dataset"), map_location=device)
    long_model = GPT(GPTConfig(**ck["cfg"])).to(device)
    long_model.load_state_dict(ck["model"])
long_model.eval()
print(f"val loss {estimate_loss(long_model, 'val'):.3f}")

### 采样方法

PPT 第 4 页最后一个问题：最终采样时只能取概率最大的那个吗？

- **greedy**：每步取概率最大的
- **temperature**：logits 除以 $\tau$ 再 softmax。$\tau < 1$ 分布更尖、更保守，$\tau > 1$ 更平、更随机
- **top-k**：只在概率最高的 $k$ 个里抽，剩下的直接排除

In [ ]:
prompt = torch.tensor([tok.encode("Once upon a time").ids], device=device)
settings = [("greedy", dict(greedy=True)),
            ("τ=0.7, top-k 50", dict(temperature=0.7, top_k=50)),
            ("τ=1.0", dict(temperature=1.0)),
            ("τ=1.5", dict(temperature=1.5))]
for name, kw in settings:
    torch.manual_seed(0)
    out = long_model.generate(prompt, 200, **kw)
    print(f"==== {name} ====\n{tok.decode(out[0].tolist())}\n")

greedy 每次都取最可能的词，很容易绕进一个循环出不来：一旦重复了一次，重复本身就让下一次重复的概率更高。$\tau = 1.5$ 则让很多不太可能的词也有机会，写着写着就不通了。

**可以试试**

- 换个开头，比如 `"Tom was sad because"`
- greedy 下数一数从第几个 token 开始出现循环
- 看看 `long_model` 和 6 节里的 `models["baseline"]`（只训了几百步）写出来的东西差在哪：语法先通，还是情节先连贯？TinyStories 论文的结论是语法先出现，前后一致要模型大一些、训得久一些才有